# Untold Lores — DynamiCrafter Animation Server
Runs DynamiCrafter (image→video) as an API. Better than SVD for anime/illustration style.

**Steps:**
1. Enable GPU: Settings → Accelerator → **GPU T4 x2**
2. Add your ngrok token to Kaggle Secrets as `NGROK_TOKEN`
3. Run all cells in order
4. Copy `SVD_NGROK_URL=...` into your `.env` file
5. Run `npm run animate` on your laptop (after `npm run images`)

In [ ]:
# Cell 1 — Install dependencies
!pip install -q --upgrade diffusers huggingface_hub transformers accelerate peft flask pyngrok imageio[ffmpeg]
!pip show diffusers huggingface_hub | grep Version

In [ ]:
# Cell 2 — Load LTX-Video
import torch
from diffusers import LTXImageToVideoPipeline
from diffusers.utils import export_to_video
from PIL import Image
import io, base64, os, tempfile

try:
    from kaggle_secrets import UserSecretsClient
    import huggingface_hub
    huggingface_hub.login(token=UserSecretsClient().get_secret("HF_TOKEN"), add_to_git_credential=False)
except Exception:
    pass

print("Loading LTX-Video...")
pipe = LTXImageToVideoPipeline.from_pretrained(
    "Lightricks/LTX-Video",
    torch_dtype=torch.bfloat16,
)
pipe.enable_model_cpu_offload()
print("LTX-Video ready!")

TMP_DIR = tempfile.mkdtemp()

In [ ]:
# Cell 3 — Start API server
from flask import Flask, request, send_file, jsonify
import threading, uuid

app = Flask(__name__)
jobs = {}

def run_ltx(job_id, image_b64, prompt, seed):
    try:
        image_bytes = base64.b64decode(image_b64)
        image = Image.open(io.BytesIO(image_bytes)).convert("RGB").resize((512, 320))
        generator = torch.Generator(device="cuda").manual_seed(seed)
        output = pipe(
            prompt=prompt,
            image=image,
            height=320,
            width=512,
            num_frames=17,
            num_inference_steps=30,
            guidance_scale=3.5,
            generator=generator,
        )
        frames = output.frames[0]
        out_path = os.path.join(TMP_DIR, f"clip_{job_id}.mp4")
        export_to_video(frames, out_path, fps=8)
        jobs[job_id] = {"status": "done", "path": out_path}
    except Exception as e:
        import traceback
        print(f"LTX error for {job_id}: {e}")
        traceback.print_exc()
        jobs[job_id] = {"status": "error", "error": str(e)}

@app.route("/health")
def health():
    return jsonify({"status": "ok"})

@app.route("/img2vid/submit", methods=["POST"])
def submit():
    data = request.json
    job_id = str(uuid.uuid4())
    jobs[job_id] = {"status": "processing"}
    t = threading.Thread(target=run_ltx, args=(
        job_id,
        data.get("image"),
        data.get("prompt", "gentle motion, natural life, subtle movement"),
        int(data.get("seed", 42)),
    ))
    t.daemon = True
    t.start()
    return jsonify({"job_id": job_id})

@app.route("/img2vid/status/<job_id>")
def status(job_id):
    job = jobs.get(job_id)
    if not job:
        return jsonify({"status": "not_found"}), 404
    return jsonify({"status": job["status"], "error": job.get("error")})

@app.route("/img2vid/result/<job_id>")
def result(job_id):
    job = jobs.get(job_id)
    if not job or job["status"] != "done":
        return jsonify({"error": "not ready"}), 400
    path = job["path"]
    del jobs[job_id]
    return send_file(path, mimetype="video/mp4")

t = threading.Thread(target=lambda: app.run(port=5000, use_reloader=False, threaded=True))
t.daemon = True
t.start()
print("Flask server started on port 5000")

In [ ]:
# Cell 4 — Expose with ngrok
from pyngrok import ngrok, conf

try:
    from kaggle_secrets import UserSecretsClient
    ngrok_token = UserSecretsClient().get_secret('NGROK_TOKEN')
except Exception:
    ngrok_token = 'PASTE_YOUR_NGROK_TOKEN_HERE'

conf.get_default().auth_token = ngrok_token

# Kill any existing tunnels before opening a new one
ngrok.kill()

public_url = ngrok.connect(5000).public_url
print('=' * 60)
print('SVD server is live!')
print(f'SVD_NGROK_URL={public_url}')
print('=' * 60)

In [ ]:
# Cell 5 — Keep session alive
import time
print('Keeping session alive... (interrupt kernel when done)')
while True:
    time.sleep(60)
    print('.', end='', flush=True)